# Tutorial 1: Vanilla calculations
This tutorial will show how to use the templates located in the `pycp2k/templates/FORCE_EVAL` directory to perform a static calculation of energies and forces. This tutorial is based on the one at https://www.cp2k.org/howto:static_calculation, but it does not perform the same calculation. Instead, it performs one calculation for each method that has a template in this repository. As such, it is continuously updated as new templates are added to the repository.

## 1. Importing Modules
In this tutorial, we will use the templates CP2K class, which wraps around the pyCP2K CP2K class and defines some of the global section variables automatically.
We will also use the ASE Atoms class to define the structure of the system.
To better illustrate the use of the templates, we will import the relevant modules when we need them.

In [1]:
import os
from copy import deepcopy

## 2. Create the CP2K object and define the structure of the system

### 2.1. Create the ASE ```Atoms``` object

The simulated atoms, their positions and their unit cell are defined in the ```Atoms``` object. In this example, we are using the ```Diamond``` class from the ```ase.lattice.cubic``` module, which generates a cubic unit cell.

In [ ]:
from ase.lattice.cubic import Diamond
atoms = Diamond(directions=[[1, 0, 0], [0, 1, 0], [0, 0, 1]],
                  symbol='Si',
                  latticeconstant=5.430697500,
                  size=(1, 1, 1))
print(atoms)

### 2.2. Create the CP2K object

Here, instead of the ```pycp2k.CP2K``` class, we use the ```pycp2k.templates.GLOBAL.GLOBAL.CP2K``` class, which is a wrapper around the ```pycp2k.CP2K``` class and defines some of the global section variables automatically, as well as the working directory and the cp2k command. There is no reason not to use the ```pycp2k.CP2K``` class, but this wrapper is useful if you want to use the templates.

In [ ]:
from pycp2k.templates.GLOBAL.GLOBAL import CP2K
calc=CP2K(cp2k_command="cp2k.psmp", #replace this with your actual cp2k commmand (e.g. "mpirun -np 4 cp2k.psmp")
          working_directory=os.getcwd(),
          project_name="energy_force_Si", #Project files will have this base name
          run_type="ENERGY_FORCE",
          print_level="MEDIUM")

## 3. The xTB_OT template
Once we have created the CP2K object, the next step is add the FORCE_EVAL section. This example uses the xTB functional with Orbital Transformation (OT) method, coded in a temmplate function in ```pycp2k.templates.FORCE_EVAL.xTB_templates.add_xTB_OT()```.
```python
from pycp2k.templates.GLOBAL.GLOBAL import CP2K
from pycp2k.templates.FORCE_EVAL.DFT.xTB import add_xTB
from pycp2k.templates.FORCE_EVAL.DFT.SCF.OT import add_OT
from pycp2k.templates.FORCE_EVAL.DFT.SCF.loop import add_inner_scf,add_outer_scf
from pycp2k.templates.FORCE_EVAL.SUBSYS.add_atoms import add_coords,add_cell
from ase import Atoms

def add_xTB_OT(atoms:Atoms,calc:CP2K,**kwargs):
    calc.CP2K_INPUT.FORCE_EVAL_add()
    add_xTB(atoms=atoms,calc=calc,**kwargs)
    add_OT(calc=calc,**kwargs)
    add_inner_scf(calc=calc,**kwargs)
    add_outer_scf(calc=calc,**kwargs)
    add_coords(atoms=atoms,calc=calc)
    add_cell(atoms=atoms,calc=calc)
    return

```
The function ```add_xTB_OT()``` takes the ```CP2K``` object and the ```Atoms``` object as arguments. When called like that, it will use default values for all arguments taken by the functions called inside it. Those arguments can be specified as keyword arguments to ```add_xTB_OT()```. 

The following code cell shows all possible keyword arguments that one can supply to ```add_xTB_OT()```.  Once the section has been added, the calculation can be run using the ```run()``` method of the ```CP2K``` class.

In [ ]:
from pycp2k.templates.FORCE_EVAL.xTB_templates import add_xTB_OT
calc_xtb_ot=deepcopy(calc)
calc_xtb_ot.CP2K_INPUT.GLOBAL.Project_name="xtb_ot"
calc_xtb_ot.project_name="xtb_ot"
add_xTB_OT(atoms=atoms, calc=calc_xtb_ot,
           feval_idx=0,
           preconditioner="FULL_ALL",minimizer="DIIS", # Options for pycp2k.templates.FORCE_EVAL.DFT.SCF.OT.add_OT
           scf_guess="RESTART",max_scf=20,eps_scf=1e-6, # Options for pycp2k.templates.FORCE_EVAL.DFT.SCF.loop.add_inner_SCF
           outer_max_scf=2,outer_eps_scf=1e-6, # Options for pycp2k.templates.FORCE_EVAL.DFT.SCF.loop.add_outer_SCF
           #FIXME: Add option to run with non-cubic cells and no PBC # Options for pycp2k.templates.FORCE_EVAL.SUBSYS.add_atoms.add_cell
           )
calc_xtb_ot.run()

# 4. The PBE_OT template
In this example, we are using the PBE functional with Orbital Transformation (OT) method, coded in a temmplate function in ```pycp2k.templates.FORCE_EVAL.PBE_templates.add_PBE_OT()```.
```python
from pycp2k.templates.GLOBAL.GLOBAL import CP2K
from pycp2k.templates.FORCE_EVAL.DFT.PBE import add_PBE
from pycp2k.templates.FORCE_EVAL.DFT.SCF.OT import add_OT
from pycp2k.templates.FORCE_EVAL.DFT.SCF.loop import add_inner_scf,add_outer_scf
from pycp2k.templates.FORCE_EVAL.DFT.SCF.mixing import add_mixing
from pycp2k.templates.FORCE_EVAL.DFT.SCF.smear import add_smear
from pycp2k.templates.FORCE_EVAL.SUBSYS.add_atoms import add_coords,add_cell
from pycp2k.templates.FORCE_EVAL.SUBSYS.add_kinds import add_kinds
from ase import Atoms

def add_PBE_OT(atoms:Atoms,calc:CP2K,**kwargs):
    calc.CP2K_INPUT.FORCE_EVAL_add()
    add_PBE(calc=calc,**kwargs)
    add_OT(calc=calc,**kwargs)
    add_inner_scf(calc=calc,**kwargs)
    add_outer_scf(calc=calc,**kwargs)
    add_coords(atoms=atoms,calc=calc)
    add_cell(atoms=atoms,calc=calc)
    add_kinds(atoms=atoms,calc=calc,**kwargs)
    return
```

```add_PBE_OT()``` only requires the ```CP2K``` object and the ```Atoms``` object as arguments. When called like that, it will use default values for all arguments taken by the functions called inside it. Those arguments can be specified as keyword arguments to ```add_PBE_OT()```. With PBE, we need to specify the potential and basis set, so we use the ```add_kinds()``` function.

The following code cell shows all possible keyword arguments that one can supply to ```add_PBE_OT()```.
Once the section has been added, the calculation can be run using the ```run()``` method of the ```CP2K``` class.

In [ ]:
from pycp2k.templates.FORCE_EVAL.PBE_templates import add_PBE_OT
calc_pbe_ot=deepcopy(calc)
calc_pbe_ot.CP2K_INPUT.GLOBAL.Project_name="pbe_ot"
calc_pbe_ot.project_name="pbe_ot"
add_PBE_OT(atoms=atoms, calc=calc_pbe_ot,
           feval_idx=0,
           potential_file_name="POTENTIAL",basis_set_files=["BASIS_MOLOPT"], # Options for pycp2k.templates.FORCE_EVAL.DFT.PBE.add_PBE
           preconditioner="FULL_ALL",minimizer="DIIS", # Options for pycp2k.templates.FORCE_EVAL.DFT.SCF.OT.add_OT
           scf_guess="RESTART",max_scf=20,eps_scf=1e-6, # Options for pycp2k.templates.FORCE_EVAL.DFT.SCF.loop.add_inner_SCF
           outer_max_scf=2,outer_eps_scf=1e-6, # Options for pycp2k.templates.FORCE_EVAL.DFT.SCF.loop.add_outer_SCF
           #FIXME: Add option to run with non-cubic cells and no PBC # Options for pycp2k.templates.FORCE_EVAL.SUBSYS.add_atoms.add_cell
           potential="GTH-PBE", basis_set=["DZVP-MOLOPT-SR-GTH"] # Options for pycp2k.templates.FORCE_EVAL.SUBSYS.add_kinds.add_kinds
           )
calc_pbe_ot.run()

# 5. The PBE_mixing_smear template
In this example, we are using the PBE functional with Mixing and Smeearing in order to improve the convergence of the SCF. This is coded tin a template function in ```pycp2k.templates.FORCE_EVAL.PBE_templates.add_PBE_mixing_smear()```.
```python
def add_PBE_mixing_smear(atoms:Atoms,calc:CP2K,**kwargs):
    calc.CP2K_INPUT.FORCE_EVAL_add()
    add_PBE(calc=calc,**kwargs)
    add_mixing(calc=calc,**kwargs)
    add_smear(calc=calc,**kwargs)
    add_coords(atoms=atoms,calc=calc)
    add_cell(atoms=atoms,calc=calc)
    add_kinds(atoms=atoms,calc=calc,**kwargs)
    return
```

```add_PBE_mixing_smear()``` only requires the ```CP2K``` object and the ```Atoms``` object as arguments. When called like that, it will use default values for all arguments taken by the functions called inside it. Those arguments can be specified as keyword arguments to ```add_PBE_OT()```. 

The following code cell shows all possible keyword arguments that one can supply to ```add_PBE_mixing_smear()```.
Once the section has been added, the calculation can be run using the ```run()``` method of the ```CP2K``` class.

In [ ]:
from pycp2k.templates.FORCE_EVAL.PBE_templates import add_PBE_mixing_smear
calc_pbe_mixing_smear=deepcopy(calc)
calc_pbe_mixing_smear.CP2K_INPUT.GLOBAL.Project_name="pbe_mixing_smear"
calc_pbe_mixing_smear.project_name="pbe_mixing_smear"
add_PBE_mixing_smear(atoms=atoms, calc=calc_pbe_mixing_smear,
           feval_idx=0,
           potential_file_name="POTENTIAL",basis_set_files=["BASIS_MOLOPT"], # Options for pycp2k.templates.FORCE_EVAL.DFT.PBE.add_PBE
           MIXING_Method="BROYDEN_MIXING",MIXING_Alpha=0.6,MIXING_Beta=1.0,MIXING_Nbroyden=15, # Options for pycp2k.templates.FORCE_EVAL.DFT.SCF.mixing.add_mixing
           SMEAR_Method="FERMI_DIRAC",SMEAR_Electronic_temperature=300.0, # Options for pycp2k.templates.FORCE_EVAL.DFT.SCF.smear.add_smear
           scf_guess="RESTART",max_scf=20,eps_scf=1e-6, # Options for pycp2k.templates.FORCE_EVAL.DFT.SCF.loop.add_inner_SCF
           outer_max_scf=2,outer_eps_scf=1e-6, # Options for pycp2k.templates.FORCE_EVAL.DFT.SCF.loop.add_outer_SCF
           #FIXME: Add option to run with non-cubic cells and no PBC # Options for pycp2k.templates.FORCE_EVAL.SUBSYS.add_atoms.add_cell
           potential="GTH-PBE", basis_set=["DZVP-MOLOPT-SR-GTH"] # Options for pycp2k.templates.FORCE_EVAL.SUBSYS.add_kinds.add_kinds
           )
calc_pbe_mixing_smear.run()

# 6. The Print template
So, running calculations is great, but at some point we will have to do something with the results. This is where the ```pycp2k.templates.PRINT``` module comes in. The structure of the ```pycp2k.templates.PRINT``` follows a pattern similar to the howtos/ folder in the pycp2k repository, as in for each notebook, there will be a PRINT template to which controls the output of the calculation and postprocessing of the results. Here, we will use the ```pycp2k.templates.PRINT.singlepoint``` module to print and postprocess the energy,forces and stress tensor. We will store those results in the original ```ase.Atoms``` object we created.





In [ ]:
from pycp2k.templates.PRINT.singlepoint import *
#Energy is printed in stdout by default, so we don't need a print function for that
forces_path=add_print_singlepoint_forces(calc=calc_xtb_ot,filename="forces")
stress_path=add_print_stress_tensor(calc=calc_xtb_ot,filename="./")
calc_xtb_ot.run()
atoms.info["E"]=postprocess_energy(calc=calc_xtb_ot)
atoms.set_array("forces",postprocess_forces(forces_path=forces_path))
atoms.info["stress"]=stress_tensor=postprocess_stress(stress_path=stress_path, notation="voigt")



